# 【完全恒久・完全固定版】藤原紀香 AI音声モデル（Style-Bert-VITS2 JP-Extra）学習パイプライン

Google Colab や Style-Bert-VITS2 の将来の改変・バージョンアップに一切影響されない**完全固定・完全隔離環境**です。
- **Colab更新耐性**: `uv` により Python 3.10 を完全隔離構築（Colab のシステム Python 3.13+ を 100% 遮断）
- **リポジトリ改編耐性**: Style-Bert-VITS2 公式安定版タグ `2.7.0` にピン留め（master改変の影響を遮断）
- **ライブラリ更新耐性**: PyTorch 2.2.2 / Transformers 4.40.2 / NumPy 1.26.4 / Setuptools <80 を厳格固定

---
### 実行手順（1クリック・手戻りゼロ）
1. 上部メニュー「ランタイム」>「ランタイムのタイプを変更」で **T4 GPU** を選択。
2. 下のコードセルの再生ボタン（▶）を 1 回クリックするだけ。
※ 約15分で学習が完走し、完成モデル (`norika_official_model.zip`) が自動ダウンロードされます。

In [ ]:
# ==============================================================================
# 【完全恒久・完全固定版】藤原紀香 Style-Bert-VITS2 JP-Extra ワンクリック学習パイプライン
# ==============================================================================
# 根本方針:
# 1. Colab システム更新耐性: uv により Python 3.10 完全隔離環境を自動構築。
# 2. Style-Bert-VITS2 改編耐性: 安定版タグ「2.7.0」に完全ピン留め（master改変の影響を遮断）。
# 3. 依存ライブラリ更新耐性: PyTorch 2.2.2 / Transformers 4.40 / NumPy 1.26.4 / Setuptools <80 (pkg_resources完備) を厳格固定。
# ==============================================================================

import os, sys, shutil, zipfile, yaml, subprocess, urllib.request
from pathlib import Path

# --- 1. 完全校正済みデータセットの自動取得 ---
%cd /content
zip_target = Path("/content/true_norika_colab_dataset.zip")
dataset_url = "https://raw.githubusercontent.com/moshajmoshaj/norika-voice-dataset/main/true_norika_colab_dataset.zip"

if zip_target.exists() and zip_target.stat().st_size < 15000000:
    print(f"🗑️ 破損・不完全な既存ファイル（{zip_target.stat().st_size} bytes）を削除...")
    zip_target.unlink()

if not zip_target.exists():
    print("📥 最新の完全校正済みデータセット（誤字ゼロ版・20MB）を高速ダウンロード中...")
    urllib.request.urlretrieve(dataset_url, str(zip_target))

if not zip_target.exists() or zip_target.stat().st_size < 15000000:
    !curl -L -f -o /content/true_norika_colab_dataset.zip "https://raw.githubusercontent.com/moshajmoshaj/norika-voice-dataset/main/true_norika_colab_dataset.zip"

assert zip_target.exists() and zip_target.stat().st_size > 15000000, f"データセット取得失敗: {zip_target.stat().st_size if zip_target.exists() else 0} bytes"
print(f"✅ 完全校正済みデータセット確認完了！ ({zip_target.stat().st_size/1024/1024:.2f} MB)")

# --- 2. 高速パッケージマネージャ uv ＆ CMake の配備 ---
os.environ["PATH"] = "/root/.cargo/bin:" + os.environ["PATH"]
os.environ["CMAKE_POLICY_VERSION_MINIMUM"] = "3.5"
if not os.path.exists("/usr/local/bin/uv") and not os.path.exists("/root/.cargo/bin/uv"):
    !curl -LsSf https://astral.sh/uv/install.sh | sh
!pip install -q "cmake<3.31"

# --- 3. Style-Bert-VITS2 専用 Python 3.10 仮想環境の独立構築（Colab更新を完全遮断） ---
VENV_DIR = "/content/sbv2_env"
PY_BIN = f"{VENV_DIR}/bin/python"

if not os.path.exists(f"{VENV_DIR}/bin/python"):
    print("📦 Python 3.10 完全隔離クリーン環境を構築中（約15秒）...")
    !uv venv --python 3.10 {VENV_DIR}

# --- 4. Style-Bert-VITS2 公式安定版 2.7.0 固定取得 ＆ 黄金依存関係の導入 ---
if not os.path.exists("/content/Style-Bert-VITS2"):
    print("📥 Style-Bert-VITS2 安定版 (ver 2.7.0) を取得中...")
    !git clone -b 2.7.0 --depth 1 https://github.com/litagin02/Style-Bert-VITS2.git /content/Style-Bert-VITS2

%cd /content/Style-Bert-VITS2

print("⚡ 黄金動作バージョン（PyTorch 2.2.2 CUDA 12.1 / Transformers 4.40 / NumPy 1.26.4 / Setuptools <80）を高速同期中...")
# PyTorch 2.2.2 + CUDA 12.1 (AudioMetaData 完備・T4 GPU ネイティブアクセラレーション)
!uv pip install --python {PY_BIN} "torch==2.2.2" "torchaudio==2.2.2" "torchvision==0.17.2" --index-url https://download.pytorch.org/whl/cu121 --no-progress
# Style-Bert-VITS2 公式依存関係
!CMAKE_POLICY_VERSION_MINIMUM=3.5 uv pip install --python {PY_BIN} -r requirements-colab.txt --no-progress
# エコシステム整合ピン留め（pkg_resources完備のsetuptools<80、use_auth_token, is_offline_mode, Numba, Librosa 100% 互換）
!uv pip install --python {PY_BIN} "transformers==4.40.2" "huggingface-hub==0.23.2" "numpy==1.26.4" "scipy==1.11.4" "setuptools<80" "tensorboard" --no-progress

# デフォルトモデルのダウンロード（BERT等）
!{PY_BIN} initialize.py --skip_default_models

# --- 5. 事前健全性厳格検証（Pre-flight Health Checks: 全5項目フェイルファスト検証） ---
print("🔍 隔離環境の事前整合性を自動検証中...")
checks = [
    ("1/5 Python バージョン隔離", "import sys; assert sys.version_info[:2] == (3, 10), f'Python 3.10必須: {sys.version}'"),
    ("2/5 T4 GPU (CUDA) 開通", "import torch; assert torch.cuda.is_available(), 'T4 GPUが認識されていません。ランタイムのタイプをGPUに変更してください。'"),
    ("3/5 音声処理基盤 (NumPy / Librosa / Numba / pkg_resources)", "import numpy, numba, librosa; assert numpy.__version__ == '1.26.4'"),
    ("4/5 音響基盤 (TorchAudio / AudioMetaData)", "import torchaudio; assert hasattr(torchaudio, 'AudioMetaData')"),
    ("5/5 特徴抽出 (Pyannote Audio Model 実機ロード)", "from pyannote.audio import Model; m = Model.from_pretrained('pyannote/wespeaker-voxceleb-resnet34-LM')")
]
for name, code in checks:
    res = subprocess.run([PY_BIN, "-c", code], capture_output=True, text=True)
    if res.returncode != 0:
        print(f"❌ {name} の検証に失敗しました:\n{res.stderr}")
        raise RuntimeError(f"Pre-flight check failed: {name}")
    print(f"  ✅ {name} 正常")
print("🎯 全コンポーネントの事前整合性確認完了！学習へ進みます。")

# --- 6. データセットの自動展開と配置 ---
!rm -rf /content/temp_dataset
with zipfile.ZipFile(str(zip_target), 'r') as z:
    z.extractall("/content/temp_dataset")

temp_dir = Path("/content/temp_dataset")
with open(temp_dir / "esd.list", "r", encoding="utf-8") as f:
    model_name = f.readline().strip().split("|")[1]
print(f"検出されたモデル名: {model_name}")

data_dir = Path(f"Data/{model_name}")
if data_dir.exists():
    shutil.rmtree(data_dir)
raw_dir = data_dir / "raw"
raw_dir.mkdir(parents=True, exist_ok=True)

for f in (temp_dir / "wavs").glob("*.wav"):
    shutil.copy(f, raw_dir / f.name)
shutil.copy(temp_dir / "esd.list", data_dir / "esd.list")

with open("configs/paths.yml", "w", encoding="utf-8") as f:
    yaml.dump({"dataset_root": "/content/Style-Bert-VITS2/Data", "assets_root": "/content/Style-Bert-VITS2/model_assets"}, f)

print("✅ データセット配置完了！")

# --- 7. 日本語特化前処理 ＆ ファインチューニングの実行 (120 Epochs) ---
pipeline_code = f'''
import yaml, subprocess, sys
from pathlib import Path
from gradio_tabs.train import preprocess_all, get_path
from style_bert_vits2.nlp.japanese import pyopenjtalk_worker

model_name = "{model_name}"
print("🔄 日本語BERT特徴量抽出・スタイルベクトル生成中...")
pyopenjtalk_worker.initialize_worker()
preprocess_all(
    model_name=model_name,
    batch_size=4,
    epochs=120,
    save_every_steps=1000,
    num_processes=2,
    normalize=False,
    trim=False,
    freeze_EN_bert=False,
    freeze_JP_bert=False,
    freeze_ZH_bert=False,
    freeze_style=False,
    freeze_decoder=False,
    use_jp_extra=True,
    val_per_lang=0,
    log_interval=200,
    yomi_error="skip",
)

# スタイルベクトル生成成否の厳格アサーション
style_vec_files = list(Path(f"Data/{{model_name}}").glob("*.npy"))
if len(style_vec_files) == 0:
    raise RuntimeError("【エラー】スタイルベクトル (*.npy) が生成されていません。前処理が異常終了しました。")
print(f"✅ スタイルベクトル生成確認成功: {{len(style_vec_files)}} files")
print("✅ 前処理完了！")

paths = get_path(model_name)
with open("default_config.yml", "r", encoding="utf-8") as f:
    yml_data = yaml.safe_load(f)
yml_data["model_name"] = model_name
with open("config.yml", "w", encoding="utf-8") as f:
    yaml.dump(yml_data, f, allow_unicode=True)

print("🚀 T4 GPU での日本語特化ファインチューニングを開始します（約15分）...")
cmd = [
    sys.executable, "train_ms_jp_extra.py",
    "--config", str(paths.config_path),
    "--model", str(paths.dataset_path),
    "--assets_root", "/content/Style-Bert-VITS2/model_assets"
]
subprocess.run(cmd, check=True)
print("🎉 学習完了！")
'''

with open("run_pipeline.py", "w", encoding="utf-8") as f:
    f.write(pipeline_code)

!{PY_BIN} run_pipeline.py

# --- 8. 完成モデルの自動ダウンロード（フェイルセーフ検証付き） ---
from google.colab import files
out_models = list(Path(f"/content/Style-Bert-VITS2/model_assets/{model_name}").glob("*.safetensors"))
assert len(out_models) > 0, f"【重大エラー】学習モデルファイル (*.safetensors) が生成されていません。上のログを確認してください。"
print(f"✅ 学習モデルファイル確認成功: {[f.name for f in out_models]}")
print("📦 完成モデル（norika_official_model.zip）をパッケージング中...")
!zip -r /content/norika_official_model.zip model_assets/
print("⬇️ ブラウザからダウンロードを開始します...")
files.download("/content/norika_official_model.zip")
print("✨ すべて完了しました！ダウンロードされた zip を手元 PC の models/norika_vits/ に解凍配置してください。")
